# M7 · Visualización y ética de datos

**Bootcamp de Datos para Funcionarios Públicos — Formación Pública**
Módulo compartido · **Cierre del tronco común** (Semana 8) · Se ejecuta en Google Colab.

## ¿Qué vas a lograr?
Ya sabes traer, limpiar, consultar y resumir datos. Falta lo que de verdad **convence**: mostrarlos. Un buen gráfico comunica en segundos lo que una tabla esconde. Pero los gráficos también pueden **mentir** —a veces sin querer—, y trabajar con datos públicos exige **responsabilidad**. En este módulo final aprenderás a hacer gráficos claros, a detectar gráficos engañosos y a manejar datos con ética.

**Competencia de salida:** crear gráficos de barras y de líneas correctos y bien rotulados, reconocer gráficos que distorsionan, y aplicar principios éticos al trabajar con datos del Estado.

**Entregable:** que las cinco celdas de chequeo muestren ✅.

> 🎓 **Este módulo cierra el tronco común.** Al terminarlo obtienes el **hito intermedio** del bootcamp: ya eres capaz de consultar datos con SQL, limpiarlos, resumirlos y visualizarlos.


## Los datos que vamos a usar
Último cambio de tema: **energía**. Usaremos la **matriz eléctrica de Chile**, una historia real y muy visual: el país pasó de depender del carbón a liderar en energías renovables.

> **Fuente:** Coordinador Eléctrico Nacional (CEN) y Generadoras de Chile. En 2024 la generación total del Sistema Eléctrico Nacional fue de unos 85.519 GWh. Trabajamos con dos miradas reales: la participación **por fuente** en 2024, y la evolución de la participación **renovable** entre 2022 y 2024 (cifras 2024 preliminares).

Tendremos dos tablas: `fuentes` (cada fuente y su % de la generación en 2024) y `renovable` (el % renovable por año).


## Preparar los datos y la librería de gráficos
Para visualizar usaremos **matplotlib**, la librería de gráficos más usada en Python (ya viene instalada en Colab). Se importa como `plt`. Ejecuta la celda de abajo para preparar el entorno: carga las dos tablas (`generacion_fuentes.csv` y `generacion_renovable.csv`) y deja todo listo.

*(Nota: Para este módulo, los archivos `generacion_fuentes.csv` y `generacion_renovable.csv` ya vienen guardados de forma fija y estática en la carpeta de tu proyecto. Fuente de los datos: Coordinador Eléctrico Nacional (CEN) y Generadoras de Chile, año 2024)*

In [ ]:
import os
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt

# Si los archivos no existen localmente (ej: en Colab), intentamos descargarlos desde GitHub
for archivo in ["generacion_fuentes.csv", "generacion_renovable.csv"]:
    if not os.path.exists(archivo):
        try:
            url = f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/A6-visualizacion-y-etica/{archivo}"  # Reemplazar por la URL real del repositorio al publicar
            urllib.request.urlretrieve(url, archivo)
            print(f"{archivo} descargado automáticamente.")
        except Exception:
            print(f"Aviso: No se pudo descargar {archivo} automáticamente. Si estás en Colab, asegúrate de subirlo manualmente.")

# Cargar la participación por fuente en la generación eléctrica, 2024 (% del total)
fuentes = pd.read_csv("generacion_fuentes.csv")

# Cargar la participación de energías renovables en la generación, por año (%)
renovable = pd.read_csv("generacion_renovable.csv")

print("Tabla 'fuentes':")
print(fuentes)
print("\nTabla 'renovable':")
print(renovable)

## 1. ¿Por qué visualizar?
Mira la tabla `fuentes`: son cinco números. Puedes leerlos, pero ¿cuál es la fuente dominante?, ¿cuánto más grande que las otras? Un **gráfico** responde de un vistazo. El ojo humano compara **alturas y posiciones** mucho más rápido que cifras en una grilla.

La regla básica: **cada tipo de pregunta tiene su gráfico**.
- Comparar **categorías** (fuentes de energía, regiones, rubros) → **gráfico de barras**.
- Ver una **evolución en el tiempo** (años, meses) → **gráfico de líneas**.
- Mostrar **partes de un todo** → a veces torta, pero con cuidado (es difícil de leer con muchas categorías; suele ser mejor una barra).


## 2. Gráfico de barras: comparar categorías
El gráfico de barras compara valores entre categorías: una barra por categoría, y la altura es el valor. Con matplotlib se hace con `ax.bar(categorias, valores)`.

Fíjate en el patrón que usaremos siempre:
1. `fig, ax = plt.subplots()` crea una figura (`fig`) y sus ejes (`ax`, donde se dibuja).
2. `ax.bar(...)` dibuja las barras.
3. `ax.set_title(...)`, `ax.set_xlabel(...)`, `ax.set_ylabel(...)` ponen título y etiquetas.
4. `plt.show()` muestra el gráfico.


In [ ]:
fig, ax = plt.subplots()
ax.bar(fuentes["fuente"], fuentes["porcentaje"])
ax.set_title("Generación eléctrica por fuente, Chile 2024")
ax.set_xlabel("Fuente")
ax.set_ylabel("% de la generación")
plt.show()

## 3. Gráfico de líneas: evolución en el tiempo
El gráfico de líneas muestra cómo cambia un valor a lo largo del tiempo. Une los puntos en orden, y la pendiente cuenta la historia (sube, baja, se estanca). Se hace con `ax.plot(x, y)`. El `marker="o"` agrega un punto en cada dato para que se vean.


In [ ]:
fig, ax = plt.subplots()
ax.plot(renovable["anio"], renovable["pct_renovable"], marker="o")
ax.set_title("Participación renovable en la generación, Chile")
ax.set_xlabel("Año")
ax.set_ylabel("% renovable")
ax.set_xticks(renovable["anio"])   # mostrar solo los años reales (2022, 2023, 2024)
plt.show()

## 4. La anatomía de un buen gráfico
Un gráfico sin contexto no comunica: confunde. Todo gráfico que entregues debería tener:
- **Título** que diga qué muestra.
- **Ejes etiquetados**, con **unidades** (%, GWh, personas...).
- **Escala honesta** (lo veremos en la sección de ética).
- **Fuente** de los datos (de dónde salieron y de qué año).

Un gráfico bien rotulado se entiende **solo**, sin que tengas que explicarlo al lado.


## 5. Gráficos que mienten (y cómo detectarlos)
Aquí entra la **ética**. El truco más común para engañar es **cortar el eje Y** para que un cambio pequeño parezca enorme. Veamos el mismo dato —la participación renovable 2022–2024— de dos formas: una honesta (eje desde 0) y una engañosa (eje recortado).


In [ ]:
fig, (ax_ok, ax_mal) = plt.subplots(1, 2, figsize=(11, 4))

# Honesto: eje Y desde 0
ax_ok.plot(renovable["anio"], renovable["pct_renovable"], marker="o")
ax_ok.set_ylim(0, 100)
ax_ok.set_title("HONESTO (eje Y desde 0)")
ax_ok.set_xlabel("Año"); ax_ok.set_ylabel("% renovable")
ax_ok.set_xticks(renovable["anio"])

# Engañoso: eje Y recortado para exagerar la subida
ax_mal.plot(renovable["anio"], renovable["pct_renovable"], marker="o", color="crimson")
ax_mal.set_ylim(54, 72)
ax_mal.set_title("ENGAÑOSO (eje Y recortado)")
ax_mal.set_xlabel("Año"); ax_mal.set_ylabel("% renovable")
ax_mal.set_xticks(renovable["anio"])

plt.tight_layout()
plt.show()

Los **mismos datos**: a la izquierda el avance se ve real pero moderado; a la derecha parece un salto espectacular, solo porque el eje no empieza en 0. **Cómo defenderte:** ante cualquier gráfico, mira primero **dónde empieza el eje** y **qué rango** abarca. Otros trucos a vigilar: elegir solo los años que convienen (*cherry-picking*), mezclar escalas, o usar imágenes de tamaños desproporcionados.


## 6. Ética de datos en el sector público
Visualizar es solo una parte. Trabajar con datos del Estado conlleva responsabilidades:

- **Privacidad.** Nunca expongas datos que permitan identificar a una persona (nombres, RUT, direcciones, datos de salud). Trabaja con datos **agregados** y anonimizados.
- **Cita la fuente.** Indica de dónde salieron los datos y de qué fecha. Da transparencia y permite verificar.
- **No induzcas a error.** Escalas honestas, contexto completo, no esconder lo que incomoda. Un gráfico del Estado debe informar, no convencer a la fuerza.
- **Conoce los límites de tus datos.** Recuerda lo aprendido: una **muestra** no representa a todo el país (M6); los datos **autodeclarados** pueden venir sucios o sesgados (M4). Sé honesto sobre lo que los datos **no** dicen.
- **Piensa en el impacto.** Detrás de cada fila hay personas. Las decisiones que se toman con tus análisis afectan vidas reales.


## ⚠️ Errores típicos de principiante
- **Gráfico sin título ni etiquetas.** Si hay que explicarlo de palabra, está incompleto.
- **Eje Y que no parte en 0** (en barras y comparaciones), lo que exagera diferencias. Salvo casos justificados, parte en 0.
- **Usar el gráfico equivocado.** Barras para categorías; líneas para tiempo. Una torta con 10 porciones es ilegible.
- **Olvidar `plt.show()`** o reusar la misma figura: dibuja todo encima. Crea una figura nueva por gráfico (`plt.subplots()`).
- **No citar la fuente** ni el año de los datos.
- **Exponer datos personales.** Trabaja siempre con datos agregados.


---
# 🛠️ Ejercicios
Cinco ejercicios con las tablas `fuentes` y `renovable`. En los de gráficos, crea la figura con `fig, axN = plt.subplots()` y dibuja sobre `axN` (así la celda de chequeo puede revisar tu gráfico). Ejecuta tu celda y luego la de chequeo (✅ / ❌).


## Ejercicio 01 — Gráfico de barras
Haz un gráfico de **barras** de la tabla `fuentes`: una barra por fuente, con su porcentaje. Ponle **título** con `ax1.set_title(...)`.

Pista: `ax1.bar(fuentes["fuente"], fuentes["porcentaje"])` y luego el título.

In [ ]:
fig, ax1 = plt.subplots()
# TODO: dibuja las barras con ax1.bar(...)
# TODO: ponle un título con ax1.set_title(...)
plt.show()

In [ ]:
try:
    assert len(ax1.patches) == 5, f"Deberías tener 5 barras (una por fuente), tienes {len(ax1.patches)}"
    assert ax1.get_title().strip() != "", "Falta el título del gráfico (ax1.set_title(...))"
    print("✅ Correcto. Ejercicio 01 superado: tu primer gráfico de barras.")
except NameError as e:
    print("❌ Falta crear el gráfico en ax1:", e)
except AssertionError as e:
    print("❌ Aún no:", e)

## Ejercicio 02 — Gráfico de líneas
Haz un gráfico de **líneas** de la tabla `renovable`: el `anio` en el eje X y `pct_renovable` en el eje Y. Ponle **título** con `ax2.set_title(...)`.

Pista: `ax2.plot(renovable["anio"], renovable["pct_renovable"], marker="o")`.

In [ ]:
fig, ax2 = plt.subplots()
# TODO: dibuja la línea con ax2.plot(...)
# TODO: ponle un título con ax2.set_title(...)
plt.show()

In [ ]:
try:
    lineas = ax2.get_lines()
    assert len(lineas) >= 1, "Falta dibujar la línea con ax2.plot(...)"
    assert len(lineas[0].get_xdata()) == 3, "La línea debería tener 3 puntos (2022, 2023, 2024)"
    assert ax2.get_title().strip() != "", "Falta el título del gráfico (ax2.set_title(...))"
    print("✅ Correcto. Ejercicio 02 superado: gráfico de líneas listo.")
except NameError as e:
    print("❌ Falta crear el gráfico en ax2:", e)
except AssertionError as e:
    print("❌ Aún no:", e)

## Ejercicio 03 — Un eje honesto
Repite el gráfico de líneas de `renovable`, pero esta vez hazlo **honesto**: el eje Y debe **empezar en 0** y llegar al menos hasta 100, para no exagerar la subida.

Pista: después de dibujar la línea, usa `ax3.set_ylim(0, 100)`.

In [ ]:
fig, ax3 = plt.subplots()
ax3.plot(renovable["anio"], renovable["pct_renovable"], marker="o")
ax3.set_title("Participación renovable (eje honesto)")
# TODO: fija el eje Y desde 0 hasta 100 con ax3.set_ylim(...)
plt.show()

In [ ]:
try:
    assert len(ax3.get_lines()) >= 1, "Falta la línea"
    y0, y1 = ax3.get_ylim()
    assert y0 == 0, "Para no exagerar, el eje Y debe partir en 0 (ax3.set_ylim(0, 100))"
    assert y1 >= 100, "El eje Y debería llegar al menos hasta 100"
    print("✅ Correcto. Ejercicio 03 superado: gráfico con escala honesta.")
except NameError as e:
    print("❌ Falta crear el gráfico en ax3:", e)
except AssertionError as e:
    print("❌ Aún no:", e)

## Ejercicio 04 — Sumar las renovables
Usando la tabla `fuentes`, calcula el **porcentaje total de fuentes renovables** (todas menos la térmica) y guárdalo en `renovable_total`. Debería coincidir con el % renovable de 2024.

Pista: suma el `porcentaje` de las filas donde `fuente` **no** es `"Térmica"`:
`fuentes.loc[fuentes["fuente"] != "Térmica", "porcentaje"].sum()`.

In [ ]:
renovable_total = 0   # TODO: suma el porcentaje de todas las fuentes menos "Térmica"

print(f"Renovable total 2024: {renovable_total}%")

In [ ]:
try:
    real = int(fuentes.loc[fuentes["fuente"] != "Térmica", "porcentaje"].sum())
    assert int(renovable_total) == real, f"Esperaba {real}, obtuve {renovable_total}"
    print("✅ Correcto. Ejercicio 04 superado: ¡leíste la matriz!")
    print(f"   En 2024, ~{real}% de la generación fue renovable y {100-real}% térmica.")
    print("   Coincide con la línea de tendencia: el dato cierra entre las dos tablas.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir renovable_total:", e)

## Ejercicio 05 — Interpretar la concentración de la matriz
No basta con calcular: hay que **interpretar**. Una matriz eléctrica puede estar **concentrada** (una sola fuente manda) o **diversificada** (el peso se reparte). Eso tiene consecuencias: si una sola fuente supera el **50%**, el sistema depende demasiado de ella y es más frágil ante una sequía, una falla o un alza de precios.

**Paso 1 — Calcula.** Usando la tabla `fuentes`, encuentra cuál es la **fuente más grande** y qué **porcentaje** tiene. Guarda el nombre en `fuente_dominante` y el porcentaje en `pct_dominante`.

Pista:
`idx = fuentes["porcentaje"].idxmax()`
`fuente_dominante = fuentes.loc[idx, "fuente"]`
`pct_dominante = int(fuentes.loc[idx, "porcentaje"])`

**Paso 2 — Interpreta.** Según el porcentaje que obtuviste, ¿qué conclusión se sigue? Elige **una** opción y guarda la letra (`"A"`, `"B"` o `"C"`) en la variable `conclusion`:

- **A.** Una sola fuente supera el 50% de la generación: la matriz está **muy concentrada** y depende en exceso de esa fuente.
- **B.** La fuente más grande está **por debajo del 50%**: ninguna fuente domina por sí sola, la matriz está **relativamente diversificada**.
- **C.** Todas las fuentes tienen exactamente el mismo porcentaje: la matriz está **perfectamente equilibrada**.

*(Opcional, no se corrige)* En una línea, ¿por qué una matriz diversificada es más robusta para el país? Escríbelo como comentario `#` en la celda de código.

In [ ]:
# Paso 1 — Calcula la fuente más grande y su porcentaje
idx = None                # TODO: usa fuentes["porcentaje"].idxmax()
fuente_dominante = None   # TODO: fuentes.loc[idx, "fuente"]
pct_dominante = None      # TODO: int(fuentes.loc[idx, "porcentaje"])

# Paso 2 — Interpreta: elige "A", "B" o "C" según el porcentaje que obtuviste
conclusion = None         # TODO: asigna la letra de la conclusión correcta

# Reflexión opcional (no se corrige): ¿por qué diversificar da más robustez?
# TODO (opcional): escribe tu idea aquí

print(f"Fuente dominante: {fuente_dominante} ({pct_dominante}%)")
print(f"Mi conclusión: {conclusion}")

In [ ]:
try:
    # Recomputamos el valor esperado desde el CSV (nunca hardcodeado)
    _idx = fuentes["porcentaje"].idxmax()
    _fuente_real = fuentes.loc[_idx, "fuente"]
    _pct_real = int(fuentes.loc[_idx, "porcentaje"])
    # La conclusión correcta se deduce del cálculo, no se fija a mano
    if _pct_real > 50:
        _letra_ok = "A"
    elif fuentes["porcentaje"].nunique() == 1:
        _letra_ok = "C"
    else:
        _letra_ok = "B"

    assert fuente_dominante == _fuente_real, (
        f"La fuente más grande es '{_fuente_real}', no '{fuente_dominante}'")
    assert int(pct_dominante) == _pct_real, (
        f"Esperaba {_pct_real}% para la fuente dominante, obtuve {pct_dominante}")
    assert str(conclusion).strip().upper() == _letra_ok, (
        "Revisa tu conclusión: con la fuente mayor en "
        f"{_pct_real}% (¿supera el 50%?), la lectura correcta es otra letra")

    print("✅ Correcto. Ejercicio 05 superado: calculaste e interpretaste.")
    print(f"   {_fuente_real} lidera con {_pct_real}%, pero no llega al 50%:")
    print("   ninguna fuente manda sola, la matriz chilena está diversificada.")
except NameError as e:
    print("❌ Falta definir fuente_dominante, pct_dominante o conclusion:", e)
except AssertionError as e:
    print("❌ Aún no:", e)

---
## ¿Terminaste? — y cierre del tronco común
Si las cinco celdas de chequeo muestran ✅, completaste **M7**… ¡y todo el **tronco común**! 🎉🎓

Aprendiste a **mostrar** datos con gráficos de barras y de líneas bien rotulados, a **detectar** gráficos que engañan (ojo siempre con el eje Y), y los **principios éticos** para trabajar con datos públicos: privacidad, cita de fuentes, honestidad y conciencia del impacto.

### Lo que ya sabes hacer
Con el tronco común completo, eres capaz de **traer datos reales del Estado, limpiarlos, consultarlos con SQL, resumirlos con estadística y visualizarlos con responsabilidad**. Ese es el **hito intermedio** del bootcamp, y por sí solo ya te hace más capaz en tu trabajo.

### El camino se bifurca
Ahora eliges tu especialización:
- **Línea A · Data Analytics** (A8–A13): SQL avanzado, modelado de KPIs, herramientas BI y dashboards para reportar y apoyar decisiones.
- **Línea B · Data Scientist** (D8–D13): machine learning, modelos predictivos y pipelines reproducibles.

Ambas arrancan profundizando **SQL**, que ya conoces. ¡Felicitaciones por llegar hasta aquí!